# LightGBM Model for TDE Classification (v4)

**v4: Conditional Tuning Architecture - config-driven tune/fixed params**

**v3: Uses Global OOF evaluation in Optuna objective instead of per-fold averaging**

In [1]:
# ==========================================
# CONFIGURATION & REPRODUCIBILITY
# ==========================================
import sys
from pathlib import Path

PROJECT_ROOT = Path().absolute().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from config_loader import get_path, get_seed, get

RANDOM_STATE = get_seed()
MODEL_NAME = 'lgb'
TUNE_MODE = get('models.lgb.tune')
N_OPTUNA_TRIALS = get('models.lgb.optuna_trials')

DATA_DIR = get_path('data_processed')
MODEL_DIR = get_path('models')
SUBMISSION_DIR = get_path('submissions')
TRAIN_PATH = DATA_DIR / 'further_train_features.parquet'
TEST_PATH = DATA_DIR / 'further_test_features.parquet'
FOLDS_PATH = DATA_DIR / 'train_folds.csv'

USE_SELECTED_FEATURES = False
FEATURES_JSON_PATH = get_path('features_selection') / 'selected_features_thresh60.json'
FEATURES_JSON_KEY = 'final_robust_features'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project Root: {PROJECT_ROOT}')
print(f'Random State: {RANDOM_STATE}')
print(f'Tune Mode: {TUNE_MODE}')
print(f'Optuna Trials: {N_OPTUNA_TRIALS}')

Project Root: d:\MALLORN Private
Random State: 15
Tune Mode: False
Optuna Trials: 150


In [2]:
import re
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.metrics import precision_recall_curve
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
print('Loading data...')
train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)
folds = pd.read_csv(FOLDS_PATH)
train = train.merge(folds[['object_id', 'kfold']], on='object_id', how='left')
print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f"Class distribution: {train['target'].value_counts().to_dict()}")

Loading data...
Train shape: (3043, 347)
Test shape: (7135, 345)
Class distribution: {0: 2895, 1: 148}


In [4]:
drop_cols = get('meta_columns')

if USE_SELECTED_FEATURES:
    print(f'Loading selected features from: {FEATURES_JSON_PATH}')
    with open(FEATURES_JSON_PATH, 'r') as f:
        features_data = json.load(f)
    selected_features = features_data[FEATURES_JSON_KEY]
    feature_cols = [c for c in selected_features if c in train.columns]
    print(f'Using {len(feature_cols)} selected features (from {len(selected_features)} in JSON)')
else:
    feature_cols = [c for c in train.columns if c not in drop_cols]
    print(f'Using all {len(feature_cols)} features')

def clean_cols(cols):
    return [re.sub(r'[^A-Za-z0-9_]+', '_', str(c)) for c in cols]

X = train[feature_cols].copy()
X.columns = clean_cols(X.columns)
y = train['target'].values
kfold = train['kfold'].values

X_test = test[feature_cols].copy()
X_test.columns = clean_cols(X_test.columns)
object_ids_test = test['object_id']

scale_pos_weight = (y == 0).sum() / (y == 1).sum()
print(f'Feature count: {len(feature_cols)}')
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

Using all 344 features
Feature count: 344
scale_pos_weight: 19.56


In [5]:
def objective(trial):
    tune_params = get('models.lgb.tune_params')
    common = get('models.lgb.common_params')
    
    params = {
        **common,
        'learning_rate': trial.suggest_float('learning_rate', *tune_params['learning_rate']),
        'num_leaves': trial.suggest_int('num_leaves', *tune_params['num_leaves']),
        'max_depth': trial.suggest_int('max_depth', *tune_params['max_depth']),
        'min_child_samples': trial.suggest_int('min_child_samples', *tune_params['min_child_samples']),
        'subsample': trial.suggest_float('subsample', *tune_params['subsample']),
        'colsample_bytree': trial.suggest_float('colsample_bytree', *tune_params['colsample_bytree']),
        'reg_alpha': trial.suggest_float('reg_alpha', *tune_params['reg_alpha'], log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', *tune_params['reg_lambda'], log=True),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', *tune_params['scale_pos_weight']),
        'random_state': RANDOM_STATE
    }
    
    oof_preds = np.zeros(len(y))
    
    for fold in range(5):
        train_idx = kfold != fold
        val_idx = kfold == fold
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        clf = lgb.LGBMClassifier(**params)
        callbacks = [lgb.early_stopping(50, verbose=False)]
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=callbacks)
        preds = clf.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = preds
    
    prec, rec, _ = precision_recall_curve(y, oof_preds)
    f1 = 2 * (prec * rec) / (prec + rec + 1e-9)
    return np.max(f1)

In [6]:
if TUNE_MODE:
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    print(f'Running Optuna with {N_OPTUNA_TRIALS} trials...')
    study = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    best_params = study.best_params.copy()
    print(f'\nBest F1 Score: {study.best_value:.4f}')
    print(f'Best params: {best_params}')
else:
    best_params = get('models.lgb.fixed_params').copy()
    print('Using fixed params from config:')
    print(best_params)

Using fixed params from config:
{'learning_rate': 0.09599145211806509, 'num_leaves': 59, 'max_depth': 9, 'min_child_samples': 78, 'subsample': 0.6221231872487066, 'colsample_bytree': 0.8637723635958693, 'reg_alpha': 0.018863417999689203, 'reg_lambda': 0.02345729586906597, 'scale_pos_weight': 38.17731911383301}


In [7]:
print('\nTraining final model with best params...')
common = get('models.lgb.common_params')
final_params = {**common, **best_params, 'random_state': RANDOM_STATE}

oof_preds = np.zeros(len(y))
test_preds = np.zeros(len(X_test))

for fold in range(5):
    train_idx = kfold != fold
    val_idx = kfold == fold
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    clf = lgb.LGBMClassifier(**final_params)
    callbacks = [lgb.early_stopping(150, verbose=False)]
    clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=callbacks)
    oof_preds[val_idx] = clf.predict_proba(X_val)[:, 1]
    test_preds += clf.predict_proba(X_test)[:, 1] / 5
    print(f'Fold {fold} complete.')

prec, rec, thresh = precision_recall_curve(y, oof_preds)
f1 = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-9)
best_thresh = thresh[np.argmax(f1)]
print(f'\nOOF F1 Score: {np.max(f1):.4f} at threshold {best_thresh:.4f}')


Training final model with best params...
Fold 0 complete.
Fold 1 complete.
Fold 2 complete.
Fold 3 complete.
Fold 4 complete.

OOF F1 Score: 0.6689 at threshold 0.2070


In [8]:
oof_df = pd.DataFrame({'object_id': train['object_id'], 'target': y, f'pred_{MODEL_NAME}': oof_preds})
oof_df.to_csv(MODEL_DIR / f'oof_{MODEL_NAME}.csv', index=False)
test_df = pd.DataFrame({'object_id': object_ids_test, f'pred_{MODEL_NAME}': test_preds})
test_df.to_csv(MODEL_DIR / f'preds_{MODEL_NAME}.csv', index=False)
print(f'\nSaved OOF predictions to: {MODEL_DIR}/oof_{MODEL_NAME}.csv')
print(f'Saved test predictions to: {MODEL_DIR}/preds_{MODEL_NAME}.csv')


Saved OOF predictions to: d:\MALLORN Private\models/oof_lgb.csv
Saved test predictions to: d:\MALLORN Private\models/preds_lgb.csv
